# 🚀 NewsRec Cloud GPU Worker & Batch Precompute (Google Colab T4 GPU)

This notebook provides **two modes**:
1. **Live Remote GPU Worker**: Serves FastAPI endpoints (`/embed`, `/rerank`) via Cloudflare Tunnel for real-time inference.
2. **Resumable Batch Precompute**: Offloads full dataset embedding and cross-encoder reranking to T4 GPU, checkpointing directly to Google Drive.

### Strict Parity & Reproducibility Guarantees:
- **Strict Version Verification**: Installs from pinned requirements and fails on startup on any mismatch.
- **Pinned Hugging Face Model Revisions**: Embedder (`a5beb1e3e68b9ab74eb54cfd186867f64f240e1a`) and Reranker (`2cfc18c9415c912f9d8155881c133215df768a70`).
- **Deterministic Seeds**: Fixed seeds (`42`) across PyTorch, NumPy, and Python.
- **fp32 Precision**: Strict IEEE 754 single precision (`np.float32`) for all stored embeddings and scores.
- **Secrets Security**: Loads `HF_TOKEN` securely from Colab Secrets.

In [ ]:
# Step 1: Install exact pinned packages & Cloudflare Tunnel binary
!pip install -q fastapi uvicorn "sentence-transformers==6.1.0" "transformers==5.19.0"
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("Dependencies & Cloudflared installed.")
print("IMPORTANT: Go to Runtime > Restart session, then run from Step 2 onwards.")


In [ ]:
# Step 2: Strict Startup Version & Environment Verification
import sys
import torch
import transformers
import sentence_transformers
import numpy as np

# Fix all seeds for deterministic outputs
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True

# Check Colab Secrets for HF_TOKEN
try:
    from google.colab import userdata
    import os
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print("Loaded HF_TOKEN securely from Colab Secrets.")
except Exception:
    pass

# Assert pinned version requirements
REQUIRED = {
    'sentence_transformers': '6.1.0',
    'transformers': '5.19.0',
}

errors = []
for pkg, expected in REQUIRED.items():
    actual = getattr(sys.modules[pkg], '__version__', None)
    if actual != expected:
        errors.append(f"{pkg}: expected {expected}, found {actual}")
np_ver = getattr(sys.modules.get('numpy'), '__version__', '')
if not np_ver.startswith('2.'):
    errors.append(f'numpy: expected 2.x, found {np_ver}')

if errors:
    raise RuntimeError("STARTUP VERSION MISMATCH DETECTED:\n" + "\n".join(errors))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"All environment checks passed! Active Device: {device} ({gpu_name})")


In [ ]:
# Step 3: Load Neural Models with Pinned HF Revisions and Token Caps
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
EMBEDDING_REVISION = "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a"
RERANKER_MODEL = "BAAI/bge-reranker-base"
RERANKER_REVISION = "2cfc18c9415c912f9d8155881c133215df768a70"

print(f"Loading Embedder ({EMBEDDING_MODEL} @ {EMBEDDING_REVISION[:8]})...")
embedder = SentenceTransformer(EMBEDDING_MODEL, revision=EMBEDDING_REVISION, device=device)
embedder.max_seq_length = 256  # 254 content tokens + 2 special tokens

print(f"Loading Reranker ({RERANKER_MODEL} @ {RERANKER_REVISION[:8]})...")
reranker = CrossEncoder(RERANKER_MODEL, revision=RERANKER_REVISION, max_length=512, device=device)
print("Both models loaded into GPU memory with pinned revisions and fp32 precision.")

In [ ]:
# Step 4: Mode A - Launch FastAPI Worker & Cloudflare Tunnel (Live Mode)
import os
import re
import time
import threading
import subprocess
from typing import List
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="NewsRec GPU Worker")

class EmbedRequest(BaseModel):
    texts: List[str]

class RerankRequest(BaseModel):
    pairs: List[List[str]]  # [[query, doc], ...]

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "embedding_model": EMBEDDING_MODEL,
        "embedding_revision": EMBEDDING_REVISION,
        "reranker_model": RERANKER_MODEL,
        "reranker_revision": RERANKER_REVISION
    }

@app.post("/embed")
def embed(req: EmbedRequest):
    if not req.texts:
        return {"vectors": []}
    vecs = embedder.encode(req.texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    # Explicit fp32 serialization
    return {"vectors": np.asarray(vecs, dtype=np.float32).tolist()}

@app.post("/rerank")
def rerank(req: RerankRequest):
    if not req.pairs:
        return {"logits": []}
    pair_tuples = [(p[0], p[1]) for p in req.pairs]
    scores = reranker.predict(pair_tuples, batch_size=32, show_progress_bar=False)
    return {"logits": np.asarray(scores, dtype=np.float32).tolist()}

# Start uvicorn in background thread
def start_server():
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

print("Connecting Cloudflare Tunnel...")
public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n" + "=" * 65)
    print("🎉 YOUR CLOUD GPU WORKER IS ONLINE!")
    print(f"REMOTE URL: {public_url}")
    print("=" * 65)
    print("\n👉 Copy this line into your local .env file:\n")
    print(f"REMOTE_MODEL_URL={public_url}\n")
else:
    print("Could not start tunnel automatically. Please re-run this cell.")

In [ ]:
# ============================================================
# Step 4b: KEEP-ALIVE LOOP -- Run this after Step 4
# ============================================================
# This cell runs FOREVER (until you manually interrupt it or
# disband the Colab runtime yourself).
#
# What it does every 4 minutes:
#   1. Pings the local /health endpoint to simulate activity
#   2. Runs a tiny GPU tensor op to register GPU activity
#   3. Prints a heartbeat so you can see it is alive
#
# HOW TO STOP: Click the [stop] button on this cell, or
#              Runtime > Disconnect and delete runtime.
# ============================================================

import urllib.request
import json as _json
import time
import datetime

HEARTBEAT_INTERVAL_SECONDS = 240  # 4 min -- well within Colab 90-min idle timeout
WORKER_LOCAL_URL = "http://127.0.0.1:8000"

print("Keep-alive loop started. This cell runs until YOU stop it.")
print(f"   Heartbeat every {HEARTBEAT_INTERVAL_SECONDS // 60} minutes.")
print("   Stop this cell to shut down. Runtime stays alive as long as this runs.\n")

heartbeat_count = 0
start_time = time.time()

while True:
    heartbeat_count += 1
    now = datetime.datetime.now().strftime("%H:%M:%S")
    elapsed_min = int((time.time() - start_time) / 60)

    # 1. Ping local FastAPI health endpoint
    health_status = "UNKNOWN"
    try:
        req = urllib.request.Request(
            f"{WORKER_LOCAL_URL}/health",
            headers={"User-Agent": "NewsRec-Keepalive"}
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = _json.loads(resp.read().decode("utf-8"))
            health_status = "OK" if data.get("status") == "ok" else "DEGRADED"
    except Exception as e:
        health_status = f"ERROR ({e})"

    # 2. Tiny GPU pulse -- keeps CUDA context warm and signals GPU usage to Colab
    gpu_info = ""
    try:
        if torch.cuda.is_available():
            _pulse = torch.zeros(64, 64, device="cuda") + 1.0
            _result = float(_pulse.sum().item())
            mem_mb = torch.cuda.memory_allocated() / 1024 / 1024
            gpu_info = f" | GPU mem: {mem_mb:.0f} MB"
            del _pulse
    except Exception:
        pass

    # 3. Heartbeat status line
    status_icon = "[OK]" if health_status == "OK" else "[!!]"
    print(
        f"{status_icon} [{now}] Heartbeat #{heartbeat_count} | "
        f"Worker: {health_status} | "
        f"Uptime: {elapsed_min}m{gpu_info}"
    )

    time.sleep(HEARTBEAT_INTERVAL_SECONDS)


In [ ]:
# ============================================================
# Step 5: CONNECTION TEST -- Run this ANYTIME to verify the
#         tunnel is properly connected to your local machine
# ============================================================
# Tests the FULL end-to-end chain:
#   Public Tunnel URL -> Cloudflare -> Colab FastAPI -> Models
#
# Paste the REMOTE_MODEL_URL from your .env file below if
# public_url is not set (i.e. you are running this cell alone).
# ============================================================

import urllib.request
import json as _json
import numpy as np
import time

# Auto-reads from Step 4 if run in order; otherwise paste your URL
TEST_URL = public_url if 'public_url' in dir() and public_url else "PASTE_YOUR_URL_HERE"

def _post(endpoint, payload_dict, timeout=30):
    payload = _json.dumps(payload_dict).encode("utf-8")
    req = urllib.request.Request(
        f"{TEST_URL.rstrip('/')}{endpoint}",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return _json.loads(resp.read().decode("utf-8"))

def _get(endpoint, timeout=10):
    req = urllib.request.Request(
        f"{TEST_URL.rstrip('/')}{endpoint}",
        headers={"User-Agent": "NewsRec-ConnectionTest"}
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return _json.loads(resp.read().decode("utf-8"))

print("=" * 60)
print(f"CONNECTION TEST against: {TEST_URL}")
print("=" * 60)

all_passed = True

# -- Test 1: /health ---------------------------------------------------
print("\n[TEST 1/3] /health endpoint...")
try:
    t0 = time.time()
    health = _get("/health")
    latency = (time.time() - t0) * 1000
    assert health.get("status") == "ok", f"Expected status=ok, got: {health}"
    print(f"  PASS -- status=ok | device={health.get('device')} | gpu={health.get('gpu')} | latency={latency:.0f}ms")
    print(f"     embed_model : {health.get('embedding_model')}")
    print(f"     rerank_model: {health.get('reranker_model')}")
    EXPECTED_EMBED_REV  = "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a"
    EXPECTED_RERANK_REV = "2cfc18c9415c912f9d8155881c133215df768a70"
    rev_embed = health.get("embedding_revision", "")
    rev_rerank = health.get("reranker_revision", "")
    print("     embed_revision  :", "MATCH" if rev_embed == EXPECTED_EMBED_REV else f"MISMATCH got {rev_embed}")
    print("     rerank_revision :", "MATCH" if rev_rerank == EXPECTED_RERANK_REV else f"MISMATCH got {rev_rerank}")
except Exception as e:
    print(f"  FAIL -- {e}")
    all_passed = False

# -- Test 2: /embed ----------------------------------------------------
print("\n[TEST 2/3] /embed endpoint (2 test sentences)...")
TEST_SENTENCES = [
    "Virat Kohli hits his 50th ODI century at Wankhede Stadium",
    "Supreme Court hears plea on electoral bond transparency"
]
try:
    t0 = time.time()
    result = _post("/embed", {"texts": TEST_SENTENCES})
    latency = (time.time() - t0) * 1000
    vectors = np.array(result["vectors"], dtype=np.float32)
    assert vectors.shape == (2, 768), f"Expected shape (2, 768), got {vectors.shape}"
    norms = np.linalg.norm(vectors, axis=1)
    assert all(abs(n - 1.0) < 0.01 for n in norms), f"Vectors not normalized! norms={norms}"
    cosine_sim = float(vectors[0] @ vectors[1])
    print(f"  PASS -- shape={vectors.shape} | dtype=float32 | latency={latency:.0f}ms")
    print(f"     Norms: {norms[0]:.4f}, {norms[1]:.4f}  (should be ~1.0)")
    print(f"     Cosine similarity Sports vs Politics: {cosine_sim:.4f}  (expect ~0.2-0.5)")
except Exception as e:
    print(f"  FAIL -- {e}")
    all_passed = False

# -- Test 3: /rerank ---------------------------------------------------
print("\n[TEST 3/3] /rerank endpoint (relevant pair + irrelevant pair)...")
RELEVANT_PAIR   = ["Kohli ODI century record Wankhede", "Virat Kohli scores 50th ODI hundred, breaks all-time record"]
IRRELEVANT_PAIR = ["Kohli ODI century record Wankhede", "Budget 2025: Finance ministry plans new tax slab revision"]
try:
    t0 = time.time()
    result = _post("/rerank", {"pairs": [RELEVANT_PAIR, IRRELEVANT_PAIR]})
    latency = (time.time() - t0) * 1000
    logits = np.array(result["logits"], dtype=np.float32)
    assert logits.shape == (2,), f"Expected shape (2,), got {logits.shape}"
    probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    print(f"  PASS -- shape={logits.shape} | latency={latency:.0f}ms")
    print(f"     Relevant pair   -- logit: {logits[0]:.4f} | prob: {probs[0]:.4f}")
    print(f"     Irrelevant pair -- logit: {logits[1]:.4f} | prob: {probs[1]:.4f}")
    sanity = "PASS" if probs[0] > probs[1] else "FAIL -- relevant scored lower than irrelevant"
    print(f"     Sanity check: {sanity}")
    if probs[0] <= probs[1]:
        all_passed = False
except Exception as e:
    print(f"  FAIL -- {e}")
    all_passed = False

# -- Final verdict -----------------------------------------------------
print("\n" + "=" * 60)
if all_passed:
    print("ALL TESTS PASSED -- Colab worker is fully connected!")
    print(f"   Set in your local .env:")
    print(f"   REMOTE_MODEL_URL={TEST_URL}")
else:
    print("SOME TESTS FAILED -- check errors above.")
    print("   Common causes:")
    print("   1. Tunnel not started  -- run Step 4 first")
    print("   2. Wrong URL           -- copy REMOTE URL printed by Step 4")
    print("   3. Runtime disconnected -- restart and re-run all steps")
print("=" * 60)


In [ ]:
# Step 5: Mode B - Resumable Batch Precompute with Google Drive Checkpoints
# Use this cell to precompute large datasets offline and export validated bundles.
import json
import hashlib
import pandas as pd
from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/newsrec_colab")
CHECKPOINT_FILE = CHECKPOINT_DIR / "checkpoint.json"
CHECKPOINT_EVERY = 50

def mount_drive_if_needed():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted. Checkpoints will be saved to: {CHECKPOINT_DIR}")
    except Exception as e:
        print(f"Notice: Drive mount skipped ({e}). Using local session storage.")
        Path("newsrec_colab").mkdir(exist_ok=True)

def run_batch_precompute(csv_path: str, config_hash: str):
    mount_drive_if_needed()
    target_file = CHECKPOINT_FILE if CHECKPOINT_DIR.exists() else Path("newsrec_colab/checkpoint.json")
    
    # Load existing checkpoint if resuming
    data = {"config_hash": config_hash, "articles": {}}
    if target_file.exists():
        try:
            with open(target_file, "r", encoding="utf-8") as f:
                saved = json.load(f)
                if saved.get("config_hash") == config_hash:
                    data = saved
                    print(f"Resuming from checkpoint: {len(data['articles'])} articles already processed.")
        except Exception:
            pass

    df = pd.read_csv(csv_path)
    print(f"Total articles in CSV: {len(df)}")
    
    # Process remaining articles
    unprocessed = [row for _, row in df.iterrows() if str(row['id']) not in data["articles"]]
    print(f"Articles to process: {len(unprocessed)}")
    
    count = 0
    for row in unprocessed:
        aid = str(row['id'])
        text_embed = str(row.get('text_embed', f"{row['title']}. {row.get('description', '')}"))
        vec = embedder.encode(text_embed, normalize_embeddings=True, show_progress_bar=False)
        
        # Store strictly in fp32
        data["articles"][aid] = {
            "embedding": np.asarray(vec, dtype=np.float32).tolist(),
            "title": str(row['title']),
        }
        count += 1
        if count % CHECKPOINT_EVERY == 0:
            with open(target_file, "w", encoding="utf-8") as f:
                json.dump(data, f)
            print(f"Checkpointed {len(data['articles'])}/{len(df)} articles to Drive.")

    # Final export
    export_file = target_file.parent / "newsrec_export.json"
    with open(export_file, "w", encoding="utf-8") as f:
        json.dump(data, f)
    print(f"Export complete! Saved to {export_file}")

print("Batch precompute function ready. Call run_batch_precompute('articles.csv', '<CONFIG_HASH>')")